#Check empty files

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [20]:
import os

folder_path = "/content/drive/MyDrive/Colab Notebooks/Eithar"

print("Checking files in:", folder_path)
print("-" * 60)

for filename in os.listdir(folder_path):
    file_path = os.path.join(folder_path, filename)

    if os.path.isfile(file_path):
        size = os.path.getsize(file_path)
        status = "EMPTY ⚠️" if size == 0 else "OK"
        print(f"{filename:40}  Size: {size:8} bytes   Status: {status}")


Checking files in: /content/drive/MyDrive/Colab Notebooks/Einaff
------------------------------------------------------------
backward_7.csv                            Size:   153819 bytes   Status: OK
forward_10.csv                            Size:   152097 bytes   Status: OK
backward_6.csv                            Size:   153844 bytes   Status: OK
backward_3.csv                            Size:   153955 bytes   Status: OK
backward_9.csv                            Size:   153510 bytes   Status: OK
forward_12.csv                            Size:   152299 bytes   Status: OK
forward_13.csv                            Size:   152000 bytes   Status: OK
backward_4.csv                            Size:   153783 bytes   Status: OK
forward_17.csv                            Size:   152147 bytes   Status: OK
forward_18.csv                            Size:   152183 bytes   Status: OK
forward_11.csv                            Size:   152146 bytes   Status: OK
forward_16.csv                        

# Weak Chanels

In [19]:
import numpy as np
import pandas as pd
from scipy.signal import butter, filtfilt, welch
import os

# -------------------------------
# Signal Filtering
# -------------------------------
def bandpass_filter(signal, fs, low=1, high=40):
    b, a = butter(4, [low/(fs/2), high/(fs/2)], btype='band')
    return filtfilt(b, a, signal)

def notch_filter(signal, fs, freq=50):
    b, a = butter(2, [(freq-2)/(fs/2), (freq+2)/(fs/2)], btype='bandstop')
    return filtfilt(b, a, signal)

# -------------------------------
# Bandpower
# -------------------------------
def bandpower(signal, fs, band):
    f, Pxx = welch(signal, fs, nperseg=min(len(signal), fs*2))
    idx = (f >= band[0]) & (f <= band[1])
    return np.trapezoid(Pxx[idx], f[idx]) if np.any(idx) else 1e-6

# -------------------------------
# Compute ERD/ERS
# -------------------------------
def compute_erd_ers(rest, imagery, fs):
    mu_band       = (7, 12)
    beta_erd_band = (13, 24)
    beta_ers_band = (25, 35)

    rest_mu  = bandpower(rest, fs, mu_band)
    img_mu   = bandpower(imagery, fs, mu_band)

    rest_beta_erd = bandpower(rest, fs, beta_erd_band)
    img_beta_erd  = bandpower(imagery, fs, beta_erd_band)

    rest_beta_ers = bandpower(rest, fs, beta_ers_band)
    img_beta_ers  = bandpower(imagery, fs, beta_ers_band)

    if min(rest_mu, rest_beta_erd, rest_beta_ers) < 1e-6:
        return None, None, None, "Invalid"

    erd_mu   = (img_mu       - rest_mu)       / rest_mu
    erd_beta = (img_beta_erd - rest_beta_erd) / rest_beta_erd
    ers_beta = (img_beta_ers - rest_beta_ers) / rest_beta_ers

    return erd_mu, erd_beta, ers_beta, "OK"

# -------------------------------
# Evaluate MI per channel
# -------------------------------
def evaluate_erd(erd_mu, erd_beta, ers_beta):
    if erd_mu is None:
        return "Invalid"
    if erd_mu < -0.10 or erd_beta < -0.10 or ers_beta > 0.10:
        return "Clear"
    if erd_mu < -0.05 or erd_beta < -0.05 or ers_beta > 0.05:
        return "Moderate"
    return "Weak"

# -------------------------------
# Analyze a single recording
# -------------------------------
def analyze_mi_recording(csv_path, fs=125):
    df = pd.read_csv(csv_path, header=None).iloc[:, 1:9]
    results = []

    for i in range(8):
        ch = df.iloc[:, i].astype(float).values

        if np.std(ch) < 1e-6:
            results.append((f"ch_{i+1}", None, None, None, "Invalid"))
            continue

        try:
            ch_f = bandpass_filter(notch_filter(ch, fs), fs)
        except:
            results.append((f"ch_{i+1}", None, None, None, "Invalid"))
            continue

        if len(ch_f) < 8 * fs:
            results.append((f"ch_{i+1}", None, None, None, "Invalid"))
            continue

        rest  = np.concatenate([ch_f[:2*fs], ch_f[6*fs:8*fs]])
        img   = ch_f[2*fs:6*fs]

        if np.std(rest) < 1e-6 or np.std(img) < 1e-6:
            results.append((f"ch_{i+1}", None, None, None, "Invalid"))
            continue

        erd_mu, erd_beta, ers_beta, flag = compute_erd_ers(rest, img, fs)
        status = "Invalid" if flag != "OK" else evaluate_erd(erd_mu, erd_beta, ers_beta)

        results.append((f"ch_{i+1}", erd_mu, erd_beta, ers_beta, status))

    return results

# -------------------------------
# Analyze folder (only required output)
# -------------------------------
def analyze_folder(folder_path, fs=125):

    weak_warnings = []        # Weak > 3 channels
    weak_main_channels = []   # Weak in C3/C4/Cz

    for filename in os.listdir(folder_path):
        if filename.endswith(".csv"):

            results = analyze_mi_recording(os.path.join(folder_path, filename), fs)

            weak_channels = [r[0] for r in results if r[4] == "Weak"]
            if len(weak_channels) > 3:
                weak_warnings.append((filename, weak_channels))

            main_weak = [r[0] for r in results if r[0] in ["ch_1", "ch_3", "ch_5"] and r[4] == "Weak"]
            if main_weak:
                weak_main_channels.append((filename, main_weak))

    print("\n==============================================")
    print("📌 Files containing Weak in more than 3 channels")
    print("==============================================")
    if not weak_warnings:
        print("✔️ No files contain Weak in more than 3 channels")
    else:
        for fname, channels in weak_warnings:
            print(f"\n🔹 File: {fname}")
            print("   Weak channels:", channels)

    print("\n==============================================")
    print("📌 Files containing Weak in main channels (C3, C4, Cz)")
    print("==============================================")
    if not weak_main_channels:
        print("✔️ No files contain Weak in C3/C4/Cz")
    else:
        for fname, channels in weak_main_channels:
            print(f"\n🔹 File: {fname}")
            print("   Weak main channels:", channels)


folder_path = "/content/drive/MyDrive/Colab Notebooks/Eithar"
analyze_folder(folder_path)



📌 Files containing Weak in more than 3 channels

🔹 File: right_11.csv
   Weak channels: ['ch_4', 'ch_5', 'ch_6', 'ch_7', 'ch_8']

🔹 File: right_4.csv
   Weak channels: ['ch_1', 'ch_4', 'ch_5', 'ch_7', 'ch_8']

🔹 File: right_6.csv
   Weak channels: ['ch_3', 'ch_4', 'ch_6', 'ch_7', 'ch_8']

🔹 File: left_17.csv
   Weak channels: ['ch_4', 'ch_5', 'ch_7', 'ch_8']

🔹 File: left_3.csv
   Weak channels: ['ch_1', 'ch_2', 'ch_3', 'ch_8']

📌 Files containing Weak in main channels (C3, C4, Cz)

🔹 File: forward_12.csv
   Weak main channels: ['ch_1']

🔹 File: forward_13.csv
   Weak main channels: ['ch_3']

🔹 File: backward_4.csv
   Weak main channels: ['ch_3']

🔹 File: forward_17.csv
   Weak main channels: ['ch_3']

🔹 File: forward_16.csv
   Weak main channels: ['ch_3']

🔹 File: backward_15.csv
   Weak main channels: ['ch_3']

🔹 File: forward_7.csv
   Weak main channels: ['ch_1']

🔹 File: left_12.csv
   Weak main channels: ['ch_5']

🔹 File: forward_5.csv
   Weak main channels: ['ch_5']

🔹 File: sto